# Suraksha Schemes: Colab Ingestion (CUDA)
Upload `chhattisgarh_chunks.jsonl` and your `.env` file to the Colab files section before running this.

In [ ]:
!pip install qdrant-client sentence-transformers python-dotenv pandas

In [ ]:
import json
import os
import time
from dotenv import load_dotenv
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient
from qdrant_client.http.models import PointStruct, VectorParams, Distance

# Load environment variables
load_dotenv()
QDRANT_URL = os.environ.get("QDRANT_CLUSTER_URL")
QDRANT_KEY = os.environ.get("QDRANT_API_KEY")
COLLECTION_NAME = "indian_schemes"

if not QDRANT_URL or not QDRANT_KEY:
    raise ValueError("Missing QDRANT_CLUSTER_URL or QDRANT_API_KEY. Did you upload the .env file?")

print("Loading JSONL Data...")
chunks = []
with open("chhattisgarh_chunks.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        chunks.append(json.loads(line))
print(f"Loaded {len(chunks)} chunks.")

print("Loading LaBSE Model on CUDA (GPU)...")
model = SentenceTransformer('sentence-transformers/LaBSE', device='cuda')

print("Connecting to Qdrant Cloud...")
client = QdrantClient(url=QDRANT_URL, api_key=QDRANT_KEY, timeout=60.0)

# Ensure collection exists
collections = [c.name for c in client.get_collections().collections]
if COLLECTION_NAME not in collections:
    client.create_collection(
        collection_name=COLLECTION_NAME,
        vectors_config=VectorParams(size=768, distance=Distance.COSINE)
    )
    print(f"Created new collection: {COLLECTION_NAME}")

batch_size = 500
start_time = time.time()
print(f"\nUploading {len(chunks)} points to Qdrant in batches of {batch_size}...")

for i in range(0, len(chunks), batch_size):
    batch_chunks = chunks[i:i+batch_size]
    texts = [c['text'] for c in batch_chunks]
    
    # Embed on GPU
    embeddings = model.encode(texts, show_progress_bar=False)
    
    batch_points = []
    for j, chunk in enumerate(batch_chunks):
        batch_points.append(PointStruct(
            id=chunk['chunk_id'],
            vector=embeddings[j].tolist(),
            payload={**chunk['metadata'], "text": chunk['text']}
        ))
        
    # Push to DB
    client.upsert(collection_name=COLLECTION_NAME, points=batch_points)
    
    done = i + len(batch_chunks)
    print(f"[{done}/{len(chunks)}] Uploaded...")

elapsed = time.time() - start_time
print(f"\n✅ All done! Uploaded {len(chunks)} chunks in {elapsed:.2f} seconds.")
